# 04.5 — FastText

**Problem it solves:** Word2Vec has no vector for OOV words, and completely ignores morphology ('run' and 'running' have unrelated vectors).

**FastText solution:** Represent each word as a bag of character n-grams. 'where' = '<where>' + '<wh' + 'whe' + 'her' + 'ere' + 're>'. The word vector = sum of its subword vectors. This means any string (even OOV) can be embedded.

---

In [ ]:
import numpy as np
import re, random
from collections import Counter, defaultdict

def tokenize(text): return re.findall(r'\b[a-z]+\b', text.lower())
def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-500,500)))

def get_char_ngrams(word: str, min_n=3, max_n=6) -> list[str]:
    """
    Get character n-grams for a word.
    Adds '<' and '>' as boundary markers.
    """
    word = f'<{word}>'  # boundary markers
    ngrams = []
    for n in range(min_n, max_n + 1):
        for i in range(len(word) - n + 1):
            ngrams.append(word[i:i+n])
    return ngrams

# Demo: what subwords does FastText use?
print("Character n-grams for 'where':")
print(get_char_ngrams('where', 3, 6))

print("\nFor 'running':")
print(get_char_ngrams('running', 3, 6))

print("\nOverlap between 'running' and 'runs':")
r1 = set(get_char_ngrams('running'))
r2 = set(get_char_ngrams('runs'))
print(f"Shared: {r1 & r2}")

In [ ]:
class FastText:
    """
    Simplified FastText: Skip-gram + subword character n-grams.
    Key difference from Word2Vec: word_vec = sum(subword_vecs).
    """
    
    def __init__(self, embed_dim=30, window=3, n_negative=5,
                 lr=0.025, n_epochs=5, min_n=3, max_n=6, bucket=2000):
        self.embed_dim = embed_dim
        self.window = window
        self.n_negative = n_negative
        self.lr = lr
        self.n_epochs = n_epochs
        self.min_n = min_n
        self.max_n = max_n
        self.bucket = bucket  # hash all ngrams into bucket slots
    
    def _hash_ngram(self, ngram: str) -> int:
        """Hash n-gram to a bucket index (fast alternative to full vocab)."""
        # FNV hash (like original FastText implementation)
        h = 2166136261
        for c in ngram:
            h ^= ord(c)
            h = (h * 16777619) & 0xFFFFFFFF
        return h % self.bucket
    
    def _get_subword_indices(self, word: str) -> list[int]:
        """Get subword embedding indices for a word."""
        ngrams = get_char_ngrams(word, self.min_n, self.max_n)
        return [self._hash_ngram(ng) for ng in ngrams]
    
    def _word_vector(self, word: str) -> np.ndarray:
        """Compute word vector = word embedding + sum of subword embeddings."""
        vec = np.zeros(self.embed_dim)
        indices = []
        
        # Word-level embedding (if in vocab)
        if word in self.word2idx:
            indices.append(('word', self.word2idx[word]))
        
        # Subword embeddings
        subword_indices = self._get_subword_indices(word)
        for si in subword_indices:
            indices.append(('subword', si))
        
        if not indices: return vec
        
        for src, i in indices:
            if src == 'word':
                vec += self.W_word[i]
            else:
                vec += self.W_subword[i]
        
        return vec / len(indices)
    
    def fit(self, corpus: list[str]):
        tokenized = [tokenize(s) for s in corpus]
        counts = Counter(w for s in tokenized for w in s)
        self.vocab = sorted(counts.keys())
        self.word2idx = {w: i for i, w in enumerate(self.vocab)}
        V = len(self.vocab)
        freqs = np.array([counts[w] for w in self.vocab])**0.75
        self.noise_dist = freqs/freqs.sum()
        
        # Two matrices: word-level and subword-level
        self.W_word    = (np.random.rand(V, self.embed_dim)-0.5)/self.embed_dim
        self.W_subword = np.zeros((self.bucket, self.embed_dim))
        self.W_out     = np.zeros((V, self.embed_dim))  # output context vectors
        
        print(f"Vocab: {V}, subword buckets: {self.bucket}")
        
        for epoch in range(self.n_epochs):
            total_loss = 0; n = 0
            for sent in tokenized:
                tokens = [w for w in sent if w in self.word2idx]
                for i, center in enumerate(tokens):
                    win = random.randint(1, self.window)
                    ctx_range = tokens[max(0,i-win):i] + tokens[i+1:i+win+1]
                    
                    # Center word vector (word + subword)
                    v_center = self._word_vector(center)
                    sw_indices = self._get_subword_indices(center)
                    w_idx = self.word2idx[center]
                    
                    for ctx in ctx_range:
                        if ctx not in self.word2idx: continue
                        ctx_idx = self.word2idx[ctx]
                        u_ctx = self.W_out[ctx_idx]
                        
                        pos_score = np.dot(v_center, u_ctx)
                        loss = -np.log(sigmoid(pos_score)+1e-10)
                        
                        grad_u = (sigmoid(pos_score)-1) * v_center
                        grad_v = (sigmoid(pos_score)-1) * u_ctx
                        
                        self.W_out[ctx_idx] -= self.lr * grad_u
                        
                        for neg_idx in np.random.choice(V, self.n_negative, p=self.noise_dist):
                            if neg_idx == ctx_idx: continue
                            u_neg = self.W_out[neg_idx]
                            ns = np.dot(v_center, u_neg)
                            self.W_out[neg_idx] -= self.lr * sigmoid(ns) * v_center
                            grad_v += sigmoid(ns) * u_neg
                            loss += -np.log(sigmoid(-ns)+1e-10)
                        
                        # Distribute gradient to word + all subword embeddings
                        n_parts = 1 + len(sw_indices)
                        self.W_word[w_idx] -= self.lr * grad_v / n_parts
                        for si in sw_indices:
                            self.W_subword[si] -= self.lr * grad_v / n_parts
                        
                        total_loss += loss; n += 1
            self.lr *= 0.9
            print(f"  Epoch {epoch+1}: loss={total_loss/max(n,1):.4f}")
        return self
    
    def get_vector(self, word):
        return self._word_vector(word)
    
    def most_similar(self, word, n=5):
        vec = self.get_vector(word)
        norm = np.linalg.norm(vec)
        if norm == 0: return []
        vecs = np.array([self.get_vector(w) for w in self.vocab])
        norms = np.linalg.norm(vecs, axis=1, keepdims=True); norms[norms==0]=1
        sims = (vecs/norms) @ (vec/norm)
        top = sims.argsort()[::-1][1:n+1]
        return [(self.vocab[i], float(sims[i])) for i in top]

corpus = [
    "running and swimming are great sports for healthy people",
    "the runner won the race quickly and efficiently",
    "machine learning models run training loops automatically",
    "the quick brown fox jumps over the lazy dog daily",
    "playing football basketball and tennis keeps people active",
    "researchers studied morphological features of words carefully",
] * 20

ft = FastText(embed_dim=25, window=3, n_epochs=8, lr=0.05, bucket=500)
ft.fit(corpus)

In [ ]:
# FastText's key advantage: OOV words
oov_words = ['jogging', 'swimmer', 'runnable', 'footballing', 'quickly']
print("Vectors for OOV words (not in training vocab):")
for word in oov_words:
    in_vocab = word in ft.word2idx
    vec = ft.get_vector(word)
    norm = np.linalg.norm(vec)
    print(f"  {word:15}  in_vocab={in_vocab}  |vec|={norm:.4f}  (non-zero = has subword representation)")

print("\nWord2Vec would return zero vectors for all of these.")
print("FastText computes them from shared character n-grams.")

## Summary

**FastText key innovations:**
1. Subword character n-grams → OOV words get non-zero vectors
2. Morphological sharing: 'run', 'running', 'runner' share subword vectors
3. Hashing trick: O(bucket_size) memory instead of O(vocab × n_gram_types)

**When to use FastText over Word2Vec:**
- Morphologically rich languages (Turkish, Finnish, German)
- Text with many OOV words (social media, medical, legal)
- Tasks where word endings matter (POS tagging)

**What it doesn't fix:**
- Still one vector per word (polysemy)
- Still no global semantics (context-independence)
- Hash collisions mean different n-grams can share a bucket

---
**Next:** 04.6 — Analogy Evaluation